Training the model

In [1]:
import pandas as pd
from pathlib import Path

aligned_dir = 'Data/processed/aligned'
stock = 'Reliance_aligned.csv'
reliance_df = pd.read_csv(Path(aligned_dir).joinpath(stock))


print(reliance_df.shape)
reliance_df.head()

(733, 8)


,news_id,headline,news_time,event_date,close_T,ret_1d,ret_2d,ret_3d
0,706cb401-2b3f-44f8-ab10-ae4f47c7feae,"Stocks to Watch Today: SBI, Aurobindo Pharma, ...",2026-02-09 02:07:00+05:30,2026-02-09,1461.599976,-0.002121,0.004858,-0.008689
1,42199003-0738-4a7b-9ca0-278b685e220f,Mukesh Ambani credits stable leadership under ...,2026-02-04 18:17:00+05:30,2026-02-05,1443.400024,0.005127,0.012609,0.010461
2,6105b445-a073-4677-8007-4249f99673e4,ONGC in talks with ExxonMobil for joint bids i...,2026-01-28 18:46:00+05:30,2026-01-29,1391.000000,0.003163,-0.000431,0.033142
3,5d1732d2-19ab-49c3-bf29-6cd360821066,"Goldman Sachs buys minor stake in Axis Bank, V...",2026-01-27 21:05:00+05:30,2026-01-28,1396.699951,-0.004081,-0.000931,-0.004511
4,04429331-0de4-4a90-88af-56c80256f1d9,Buy Reliance Industries; target of Rs 1750: Mo...,2026-01-21 12:59:00+05:30,2026-01-21,1404.599976,-0.001495,-0.013171,-0.017158


In [2]:
X = reliance_df["headline"]
y = reliance_df["ret_1d"]

In [3]:
# Splitting - not using sklearn's train_test_split - bcoz of shuffling
split = int(len(X)*0.85)

X_train = X[:split]
X_val = X[split:]

y_train = y[:split]
y_val = y[split:]

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_tokens = tokenizer(
    X_train.tolist(),
    padding = True,
    truncation = True,
    max_length = 128,
    return_tensors = "pt" # Return pytorch tensors
)

val_tokens = tokenizer(
    X_val.tolist(),
    padding = True,
    truncation = True,
    max_length = 128,
    return_tensors = "pt" # Return pytorch tensors
)

In [5]:
# Converting the labels to tensors
import torch

y_train_tensors = torch.tensor(y_train.values, dtype=torch.float32)
y_val_tensors = torch.tensor(y_val.values, dtype=torch.float32)

In [6]:
# Making the class for loading data in batches

class NewsDataLoader(torch.utils.data.DataLoader):
    def __init__(self, encodings, labels): # inputs -> tokens and labels
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {k : v[idx] for k, v in self.encodings.items()}
        '''the .items() returns a dictionary with following keys values - 
        input_ids - [[token_ids of 1st article], [...],...] each list is the token ids for each article
        attention_masks - [[attention_mask for 1st article], [...],...] each list is the attention mask meaning which of the token_ids is important for each article 
        '''
        item["label"] = self.labels[idx]
        return item
    
    def __len__(self):
        return len(self.labels)

In [7]:
# Instantiate the dataloader
train_dataset_loader = NewsDataLoader(train_tokens, y_train_tensors)
val_dataset_loader   = NewsDataLoader(val_tokens, y_val_tensors)

In [8]:
# Loading the model
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=1,
    problem_type="regression")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
device = torch.device("cuda")

device

device(type='cuda')

In [10]:
# Setting training arguments
from transformers import TrainingArguments

training_args = TrainingArguments( 
    output_dir="/results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=10, # Number of steps after which the results are logged
    eval_strategy="epoch" # Evaluate after every epoch
)

In [11]:
# Instanstiating a trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_loader,
    eval_dataset=val_dataset_loader
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.004275,0.000748
2,0.001928,0.000446
3,0.002452,0.000408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=234, training_loss=0.0049684494574609986, metrics={'train_runtime': 32.5225, 'train_samples_per_second': 57.468, 'train_steps_per_second': 7.195, 'total_flos': 53785172350224.0, 'train_loss': 0.0049684494574609986, 'epoch': 3.0})

In [13]:
trainer.save_model("stock_bert_model")
tokenizer.save_pretrained("stock_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('stock_bert_model\\tokenizer_config.json', 'stock_bert_model\\tokenizer.json')

In [14]:
preds = trainer.predict(val_dataset_loader)
y_pred = preds.predictions.squeeze()

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = mean_squared_error(y_val, y_pred)
mae  = mean_absolute_error(y_val, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.00040823081371050695
MAE: 0.015859281288099034


In [30]:
import yfinance as yf

Rel = yf.Ticker("Reliance.NS")

rel_news = Rel.news
testing = []
for i in range(0,10):
    testing.append(rel_news[i]['content']['title'])

In [31]:
# making dataloading pipeline
from torch.utils.data import Dataset, DataLoader
class InferenceDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}
    
    def __len__(self):
        return len(self.encodings["input_ids"])
    
def build_dataloader(text_list, tokenizer, batch_size=8):
    encodings = tokenizer(
    text_list,
    padding = True,
    truncation = True,
    return_tensors = "pt" # Return pytorch tensors
    )

    dataset = InferenceDataset(encodings)
    loader = DataLoader(dataset, batch_size=batch_size)
    return loader

In [32]:
testing_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

testing_loader = build_dataloader(testing, testing_tokenizer)

In [33]:
from transformers import BertForSequenceClassification

def predict(loader, model_path, device="cuda"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")

    model = BertForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    preds = []

    with torch.no_grad():
        for batch in loader:
            batch = {k : v.to(device) for k,v in batch.items()}
            output = model(**batch)
            preds.extend(output.logits.squeeze().tolist())

        return preds

In [ ]:

from transformers import BertForSequenceClassification

my_model = BertForSequenceClassification.from_pretrained('./stock_bert_model')

my_model.eval()

all_predictions = []
with torch.no_grad():
    for batch in testing_loader:
        outputs = my_model(**batch)

        logits = outputs.logits
        all_predictions.extend(logits.squeeze().tolist())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
testing

['US, China Challenge Modi’s ‘Make in India’ Factory Incentives',
 'Chevron’s Iraq And Venezuela Moves Recast Growth, Risk And Valuation Profile',
 "India's Richest Man, Mukesh Ambani, Is Taking On Elon Musk's Tesla, Mark Zuckerberg's Meta With Budget-Friendly Smart Glasses, Robots",
 "Chevron sells Venezuelan oil to India's Reliance for the first time since 2023, ship\xa0data and\xa0 sources say\xa0",
 'Bill Gates pulls out of India AI summit amid Epstein scrutiny',
 'India’s Reliance Industries to Invest $110 Billion in AI',
 'Factbox-Tech majors commit billions of dollars to India at AI summit',
 'How Subtle Valuation Shifts Are Rewriting The Story For Reliance Industries (NSEI:RELIANCE)',
 "Reliance-backed Zivame bets on India's small cities with fresh store blitz, COO says",
 "India's Reliance Retail to pilot search and discovery platform in multi-channel push"]

In [44]:
Rel_3d = Rel.history('3d')
Rel_3d

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-02-24 00:00:00+05:30,1425.300049,1433.300049,1415.000000,1428.800049,12529409,0.0,0.0
2026-02-25 00:00:00+05:30,1435.000000,1440.500000,1393.500000,1398.500000,10728792,0.0,0.0
2026-02-26 00:00:00+05:30,1398.500000,1412.900024,1391.900024,1406.800049,16682753,0.0,0.0


In [ ]:
all_predictions

[-0.00901822280138731,
 -0.00043593268492259085,
 -0.014188027940690517,
 -0.01622466742992401,
 -0.01160693820565939,
 -0.016690600663423538,
 -0.01288896705955267,
 -0.023014701902866364,
 0.010386482812464237,
 -0.002282198751345277]